In [12]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
import time
from mediapipe.tasks.python import vision

In [13]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # Большой палец
    (0, 5), (5, 6), (6, 7), (7, 8),          # Указательный палец
    (5, 9), (9, 10), (10, 11), (11, 12),     # Средний палец
    (9, 13), (13, 14), (14, 15), (15, 16),   # Безымянный палец
    (13, 17), (17, 18), (18, 19), (19, 20),  # Мизинец
    (0, 5), (5, 9), (9, 13), (13, 17), (0, 17) # Основание ладони
]

In [14]:
sticker = cv2.imread('stick.jpg')

if sticker is None:
    import numpy as np
    sticker = np.zeros((100, 100, 3), dtype=np.unit8)
    sticker[:] = (255,0 ,0)

st_h, st_w, _ = sticker.shape

In [15]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)
HandLandmarker = mp.tasks.vision.HandLandmarker

In [16]:
import math

def distance(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def is_heart(hand1, hand2):
    # кончики больших пальцев
    thumb_dist = distance(hand1[4], hand2[4])

    # кончики указательных
    index_dist = distance(hand1[8], hand2[8])

    # сердечко если пальцы близко
    if thumb_dist < 0.05 and index_dist < 0.05:
        return True

    return False

In [17]:
landmarker = HandLandmarker.create_from_options(options)

In [18]:
cap = cv2.VideoCapture(0)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Не удалось получить кадр.")
            continue

        frame = cv2.flip(frame,1)
        h, w, _ =frame.shape

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        detection_result = detector.detect(mp_image)

        timestamp_ms = int(time.time() *1000)
        results = landmarker.detect_for_video(mp_image,timestamp_ms)

        if detection_result.hand_landmarks:
            for hand_landmarks in detection_result.hand_landmarks:

                index_tip = hand_landmarks[8]
                index_pip = hand_landmarks[5]
                if results.multi_hand_landmarks:
                    if len(results.multi_hand_landmarks) == 2:
                    
                        hand1 = results.multi_hand_landmarks[0].landmark
                        hand2 = results.multi_hand_landmarks[1].landmark

                        if is_heart(hand1, hand2):
                            cv2.putText(frame, "HEART", (50, 50),
                                        cv2.FONT_HERSHEY_SIMPLEX,
                                        1, (0, 0, 255), 2)

                # if index_tip.y < index_pip.y:
                #     cx = int(index_tip.x *w)
                #     cy = int(index_tip.y *h)

                #     x_offset = cx - (st_w //2)
                #     y_offset = cy - st_h - 10

                #     if y_offset > 0 and x_offset > 0 and (x_offset + st_w) <w and (y_offset + st_h)<h:
                #         frame[y_offset : y_offset + st_h, x_offset : x_offset + st_w] = sticker
                #         cv2.putText(frame, "Gestrure DETECTED!", (50, 50), 
                #                     cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2) 

                for connection in HAND_CONNECTIONS:
                    start_idx = connection[0]
                    end_idx = connection[1]

                    start_lm = hand_landmarks[start_idx]
                    end_lm = hand_landmarks[end_idx]

                    start_point = (int(start_lm.x * w), int(start_lm.y * h))
                    end_point = (int(end_lm.x * w), int(end_lm.y * h))
                    
                    cv2.line(frame, start_point, end_point, (0, 255, 0), 2)

                for landmark in hand_landmarks:
                    cx, cy = int(landmark.x *w), int(landmark.y * h)

                    cv2.circle(frame, (cx, cy), 5, (0,0,255), cv2.FILLED)

        cv2.imshow('MediaPipe Tasks API - Hand Tracking', frame)
        cv2.imshow('Gesture Image Overlay', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'): break
        

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break


finally:
    # Этот блок выполнится всегда, даже если вы аварийно остановите ячейку в ipynb
    cap.release()
    cv2.destroyAllWindows()
    print("Камера отключена, окна закрыты.")

Камера отключена, окна закрыты.


ValueError: Task is not initialized with the video mode. Current running mode:image mode

In [30]:
import cv2
import mediapipe as mp
import math
import numpy as np # Понадобится для рисования fallback-сердца
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# === ШАГ 1: Помощники и загрузка ресурсов ===

# Функция для безопасного наложения изображения с прозрачностью (PNG)
def overlay_png_transparent(background, foreground, x_offset, y_offset):
    fg_h, fg_w, fg_c = foreground.shape
    bg_h, bg_w, bg_c = background.shape

    # Проверка границ, чтобы код не упал, если сердце выходит за экран
    if y_offset < 0 or x_offset < 0 or (x_offset + fg_w) >= bg_w or (y_offset + fg_h) >= bg_h:
        return background

    # Если у картинки есть альфа-канал
    if fg_c == 4:
        alpha = foreground[:, :, 3] / 255.0 # Получаем маску прозрачности (0..1)
        # Блендинг пикселей
        for c in range(3):
            background[y_offset:y_offset+fg_h, x_offset:x_offset+fg_w, c] = \
                (1. - alpha) * background[y_offset:y_offset+fg_h, x_offset:x_offset+fg_w, c] + \
                alpha * foreground[:, :, c]
    else:
        # Если прозрачности нет, просто заменяем (как в примере со стикером)
        background[y_offset : y_offset + fg_h, x_offset : x_offset + fg_w] = foreground[:, :, :3]
    return background

# Функция рисования fallback-сердца, если PNG-файл не найден
def draw_fallback_heart(frame, center_point):
    cx, cy = center_point
    size = 30 # Базовый размер сердца
    points = np.array([
        (cx, cy + size),                   # Низ
        (cx + size, cy - size // 2),        # Правый верх
        (cx + size // 2, cy - size - size // 3), # Правая дуга
        (cx, cy - size),                   # Центр верх
        (cx - size // 2, cy - size - size // 3), # Левая дуга
        (cx - size, cy - size // 2)         # Левый верх
    ], np.int32)
    # Сглаживаем края для красоты
    points = points.reshape((-1, 1, 2))
    cv2.fillPoly(frame, [points], (0, 0, 255)) # Рисуем закрашенное КРАСНОЕ сердце

# Загружаем стикер сердца заранее
heart_img = cv2.imread('heart.jpg', cv2.IMREAD_UNCHANGED)

if heart_img is not None:
    # ПРИНУДИТЕЛЬНО уменьшаем картинку до 100x100 пикселей
    heart_img = cv2.resize(heart_img, (300, 200))
    print("Картинка успешно загружена и сжата до 100х100!")
else:
    print("ВНИМАНИЕ: 'heart.png' не найден.")

# === ШАГ 2: Настройка детектора MediaPipe для ДВУХ рук ===
# Скачайте model 'hand_landmarker.task' и положите в ту же папку
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options, 
    num_hands=2,               # ОБЯЗАТЕЛЬНО: ищем две руки
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = detector.detect(mp_image)
        
        # === ШАГ 3: Логика жеста «Сердце» ===
        
        if detection_result.hand_landmarks and len(detection_result.hand_landmarks) == 2:
            # У нас есть две руки. Но мы не знаем, какая из них левая, какая правая.
            # Для простоты логики мы просто найдем "левую" по координате X её основания.
            
            # Сортируем landmarks по возрастанию X основания ладони (точка 0)
            sorted_hands = sorted(detection_result.hand_landmarks, key=lambda lm: lm[0].x)
            
            # Левая рука (с меньшим X на экране)
            hand_l = sorted_hands[0] 
            # Правая рука (с большим X на экране)
            hand_r = sorted_hands[1]
            
            # Получаем пиксельные координаты кончиков нужных пальцев
            def to_px(lm): return (int(lm.x * w), int(lm.y * h))
            
            index_tip_l = to_px(hand_l[8])
            thumb_tip_l = to_px(hand_l[4])
            index_tip_r = to_px(hand_r[8])
            thumb_tip_r = to_px(hand_r[4])
            
            # --- ЧЕК-ЛИСТ ГЕОМЕТРИИ СЕРДЦА ---
            
            # 1. Считаем расстояние между кончиками указательных пальцев (верх сердца)
            dist_index = math.sqrt((index_tip_l[0] - index_tip_r[0])**2 + (index_tip_l[1] - index_tip_r[1])**2)
            
            # 2. Считаем расстояние между кончиками больших пальцев (низ сердца)
            dist_thumb = math.sqrt((thumb_tip_l[0] - thumb_tip_r[0])**2 + (thumb_tip_l[1] - thumb_tip_r[1])**2)
            
            # 3. Базовый контроль: указательные пальцы должны быть ВЫШЕ больших
            y_order_correct = (index_tip_l[1] < thumb_tip_l[1]) and (index_tip_r[1] < thumb_tip_r[1])
            
            index_threshold = 40
            thumb_threshold = 40
            
            if dist_index < index_threshold and dist_thumb < thumb_threshold and y_order_correct:
                
                gesture_cx = (index_tip_l[0] + index_tip_r[0] + thumb_tip_l[0] + thumb_tip_r[0]) // 4
                gesture_cy = (index_tip_l[1] + index_tip_r[1] + thumb_tip_l[1] + thumb_tip_r[1]) // 4
                center_point = (gesture_cx, gesture_cy)
                
                if heart_img is not None:

                    st_h, st_w = heart_img.shape[:2]
                    x_offset = gesture_cx - (st_w // 2)
                    y_offset = gesture_cy - (st_h // 2) - 30 
                    frame = overlay_png_transparent(frame, heart_img, x_offset, y_offset)
                else:
                    draw_fallback_heart(frame, center_point)
                    
                cv2.putText(frame, "!", (10, 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        cv2.imshow('Heart Gesture Recognition', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'): break
            
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Камера отключена.")

Картинка успешно загружена и сжата до 100х100!
Камера отключена.


KeyboardInterrupt: 